In [ ]:
import os
import itertools
import pandas as pd
import numpy as np

# Imports apontando para a pasta 'core'
from core.ingest import load_stock_data
from core.prepare import prepare_pipeline
from core import svr
from core.analysis import run_analysis_pipeline  # Pipeline ajustado com 5 níveis e campeões
from core.graphics import run_graphics_pipeline     # Visualizações prontas para w_ratio

CSV_PATH = "stock_prices_daily.csv"
LISTA_TICKERS = ["AAPL"]#, "ABBV", "ABT"]
LISTA_KERNELS = ["linear", "rbf"]
LISTA_SPLITS = [0.8]
LISTA_C = [1.0, 10.0]
LISTA_WINDOW_RATIOS = [0.05, 0.10, 0.15]  # Eixo de variação da janela (k)
EPSILON_VAL = 0.01


def gerar_combinacoes_parametros(tickers, kernels, splits, c_values, window_ratios):
    combinacoes = list(itertools.product(tickers, kernels, splits, c_values, window_ratios))
    print(f"-> Planejamento concluído: {len(combinacoes)} cenários mapeados.")
    return combinacoes


In [6]:


def main():
    print("=== INICIANDO PIPELINE DE PREVISÃO SVR MULTIVARIADO ===\n")
    
    # 1. Ingestão de Dados Real
    try:
        df_completo = load_stock_data(tickers=LISTA_TICKERS, file_path=CSV_PATH)
    except FileNotFoundError as e:
        print(f"\n[Erro] Não foi possível carregar os dados: {e}")
        return

    # 2. Geração de Combinações do Espaço de Busca
    cenarios = gerar_combinacoes_parametros(
        LISTA_TICKERS, LISTA_KERNELS, LISTA_SPLITS, LISTA_C, LISTA_WINDOW_RATIOS
    )
    
    resultados_finais = {}
    
    # 3. Loop Multivariado para Treinamento e Predição
    print("\n=== INICIANDO MODELAGEM SVR (Varredura de Hiperparâmetros e Janelas) ===")
    for idx, (ticker, kernel, split, c_value, w_ratio) in enumerate(cenarios, start=1):
        print(f"[{idx}/{len(cenarios)}] Simulando: {ticker} | {kernel} | Split: {split} | C: {c_value} | Window: {w_ratio:.2f}")
        
        # Garante a inicialização das chaves aninhadas no dicionário de resultados
        if ticker not in resultados_finais:
            resultados_finais[ticker] = {}
        if kernel not in resultados_finais[ticker]:
            resultados_finais[ticker][kernel] = {}
        if split not in resultados_finais[ticker][kernel]:
            resultados_finais[ticker][kernel][split] = {}
        if c_value not in resultados_finais[ticker][kernel][split]:
            resultados_finais[ticker][kernel][split][c_value] = {}
            
        df_ativo = df_completo[df_completo["Ticker"] == ticker].copy()
        
        # Preparação dinâmica dos dados para o cenário corrente (split e w_ratio)
        dados_preparados = prepare_pipeline(
            df_ativo, 
            split_ratio=split, 
            window_ratio=w_ratio, 
            target_col="Close"
        )
        
        ativo_dict = dados_preparados[ticker]
        
        # Treinamento e Inferência SVR
        resultado_svr = svr.run_svr_pipeline(
            ativo_dict,
            kernel=kernel,
            C=c_value,
            epsilon=EPSILON_VAL
        )
        
        # Salva na folha (quinto nível da árvore)
        resultados_finais[ticker][kernel][split][c_value][w_ratio] = {
            "predicted": resultado_svr["predicted"],
            "answers": resultado_svr["answers"]
        }
        
    print("\n=== PIPELINE SVR CONCLUÍDO COM SUCESSO! ===")
    
    # 4. Processamento de Análise das Métricas (Calculando campeões e varrendo a nova estrutura)
    analise_consolidada = run_analysis_pipeline(resultados_finais)
    
    # 5. Geração das Tabelas Consolidadas e Gravação dos Gráficos Multivariados
    run_graphics_pipeline(analise_consolidada, output_dir="output")
    
    print("\n=== EXECUÇÃO COMPLETA FINALIZADA COM SUCESSO! ===")
    return analise_consolidada


if __name__ == "__main__":
    analise_resultados = main()

=== INICIANDO PIPELINE DE PREVISÃO SVR MULTIVARIADO ===

Lendo dados de: stock_prices_daily.csv...
Filtrando dados para os tickers: ['AAPL', 'ABBV', 'ABT']...
Filtragem concluída! Registros correspondentes: 4605
Carga final concluída! Total de registros em memória: 4605
-> Planejamento concluído: 36 cenários mapeados.

=== INICIANDO MODELAGEM SVR (Varredura de Hiperparâmetros e Janelas) ===
[1/36] Simulando: AAPL | linear | Split: 0.8 | C: 1.0 | Window: 0.05
Iniciando conversão para NumPy para os tickers: ['AAPL']
  -> Ticker 'AAPL': convertido com sucesso. Formato: (1535,)
Conversão concluída para todas as ações!

--- Processando divisões para AAPL ---
Divisão concluída (Ratio: 80.0%):
  -> Treino (0 até 1228): (1228,)
  -> Teste (1228 até 1535): (307,)
  -> Normalizando dados de AAPL...
  -> Tamanho de janela k calculado: 15 (usando 5.0% de 307 dias de teste)
  -> Aplicando janelamento no Treino...
Janelamento concluído (Janela k=15):
  -> Matriz de janelas (X): (1213, 15)
  -> Vetor